In [8]:
import json
import jwt
import time
import requests

base_url = 'https://land.copernicus.eu'

In [9]:
# Load saved key from filesystem
service_key = json.load(open('history/my_saved_key.json', 'rb'))

private_key = service_key['private_key'].encode('utf-8')

claim_set = {
    "iss": service_key['client_id'],
    "sub": service_key['user_id'],
    "aud": service_key['token_uri'],
    "iat": int(time.time()),
    "exp": int(time.time() + (60 * 60)),
}
grant = jwt.encode(claim_set, private_key, algorithm='RS256')

In [10]:
result = requests.post(
        service_key["token_uri"],
        headers={
            "Accept": "application/json",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
            "assertion": grant,
        },
)

access_token_info_json = result.json()
access_token = access_token_info_json.get('access_token')
print(access_token)

t7sQDYxGwMHnt5bNYfKm9XhioGOk8YuZxn27q8QvywSx3UkQXceClE9vjVtyTxKhqOtAMeD_evlN2lPKDQCJxA==


In [16]:
search_term = "Normalised Difference Vegetation Index"
url_find_dataset = f"{base_url}/api/@search?portal_type=DataSet&metadata_fields=UID&SearchableText={requests.utils.quote(search_term)}"
headers = {'Accept': 'application/json', 'Authorization': f'Bearer {access_token}'}
response_downloadable_prepackaged = requests.get(url_find_dataset, headers=headers)

json_downloadable_prepackaged = response_downloadable_prepackaged.json()

for item in json_downloadable_prepackaged.get('items', []):
    if 'Normalised Difference Vegetation Index 1999-2020 (raster 1 km), global' in item['title']:
        print('-' * 100)
        print('找到所需NDVI数据集: {}'.format(item['title']))
        print('-' * 100)
        print(3*' '+'Product title: "'+ item['title'].replace(':','')+'"')
        print(3*' '+'UID: "'+item['UID'].replace(' ','')+'"')
        print(3*' '+'Product link: '+ item['@id']+'')
        print('-' * 100)
    print('非所需NDVI数据集: {}'.format(item['title']))
    """
    输出为:
    """

非所需NDVI数据集: Normalised Difference Vegetation Index 2014-2020 (raster 300 m), global, 10-daily – version 1
----------------------------------------------------------------------------------------------------
找到所需NDVI数据集: Normalised Difference Vegetation Index 1999-2020 (raster 1 km), global, 10-daily – version 3
----------------------------------------------------------------------------------------------------
   Product title: "Normalised Difference Vegetation Index 1999-2020 (raster 1 km), global, 10-daily – version 3"
   UID: "7714f261ebe64372bef240232aa5219a"
   Product link: https://land.copernicus.eu/api/en/products/vegetation/normalised-difference-vegetation-index-v3-0-1km
----------------------------------------------------------------------------------------------------
非所需NDVI数据集: Normalised Difference Vegetation Index 1999-2020 (raster 1 km), global, 10-daily – version 3
非所需NDVI数据集: Normalised Difference Vegetation Index 2020-present (raster 300 m), global, 10-daily – versio